In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
vocab = ["The", "cat", "sat", "on", "the", "mat"]
V, d_model = 6, 4

embed = nn.Embedding(V, d_model)
head  = nn.Linear(d_model, V, bias=False)
nn.init.normal_(embed.weight, std=0.02)       # küçük ağırlık -> logitler ≈ 0 -> dağılım ≈ uniform
nn.init.normal_(head.weight,  std=0.02)

ids = torch.tensor([[0, 1, 2, 3, 4, 5]])      # "The cat sat on the mat", şekil (1, 6)

inputs  = ids[:, :-1]                         # (1, 5): The cat sat on the
targets = ids[:, 1:]                          # (1, 5): cat sat on the mat   <- 1 kaydırılmış
logits  = head(embed(inputs))                 # (1, 5, 6)

loss = F.cross_entropy(logits.reshape(-1, V), # (5, 6): 5 pozisyon, 6 sınıf
                       targets.reshape(-1))   # (5,)
print("logits:", tuple(logits.shape), "| loss:", round(loss.item(), 4),
      "| ln(6) =", round(torch.log(torch.tensor(6.)).item(), 4))

# Causal maske: 5 pozisyon için alt üçgen
mask = torch.tril(torch.ones(5, 5))
print(mask)

logits: (1, 5, 6) | loss: 1.7916 | ln(6) = 1.7918
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
